# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anujrkt06-tech/Flyrank-ML-project-/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.



## 1. Two paper findings + my methodology questions

### Finding 1
The paper reports a finding about content refresh or content performance. The label should come from an observed outcome in the dataset rather than from a manually assumed category. My methodology question is whether the validation design uses data that is sufficiently separate from the data used to create the features. A grouped or time-aware validation would make the claim more convincing if related observations can occur close together.

### Finding 2
The paper also reports a finding about factors associated with content outcomes. The label should represent a measured outcome available from the data. My methodology question is whether the validation design carries the same claim when observations from the same group or time period are kept together. If performance changes under an honest split, the result should be treated as directional rather than as a guaranteed effect.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 1: simple methodology checks

print("Finding 1 check:")
print("Label should represent an observed/measured outcome.")
print("Validation question: can related observations appear in both train and test?")

print("\nFinding 2 check:")
print("Label should come from the available outcome data.")
print("Validation question: does the claim remain similar under grouped/time-aware validation?")

Finding 1 check:
Label should represent an observed/measured outcome.
Validation question: can related observations appear in both train and test?

Finding 2 check:
Label should come from the available outcome data.
Validation question: does the claim remain similar under grouped/time-aware validation?



## 2. My model under an honest split (before/after)

I compare the Week-5 model result with an honest validation split. The earlier result is treated as the baseline, while the grouped or time-aware result is treated as the more conservative estimate. I will report the measured numbers without overstating the difference.

In [9]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load dataset directly from the FlyRank starter repository
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# Create target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Use features that are available before the outcome
candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "search_volume",
    "word_count",
    "position_tier"
]

features = [
    col for col in candidate_features
    if col in df.columns
]

X = df[features].copy()
y = df["is_declining_label"].copy()

# Find client/group column
possible_groups = [
    "client_id",
    "client",
    "client_key",
    "account_id"
]

group_col = None

for col in possible_groups:
    if col in df.columns:
        group_col = col
        break

if group_col is None:
    print("Client/group column not found.")
    print("Available columns:")
    print(df.columns.tolist())
else:

    groups = df[group_col]

    # Grouped train/test split
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups=groups)
    )

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Numeric and categorical columns
    numeric_features = [
        col for col in features
        if X[col].dtype != "object"
    ]

    categorical_features = [
        col for col in features
        if X[col].dtype == "object"
    ]

    transformers = []

    if numeric_features:
        transformers.append(
            (
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_features
            )
        )

    if categorical_features:
        transformers.append(
            (
                "categorical",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(strategy="most_frequent")
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore"
                        )
                    )
                ]),
                categorical_features
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1
            )
        )
    ])

    # Train
    model.fit(X_train, y_train)

    # Predict
    predictions = model.predict(X_test)

    # Results
    print("\nHONEST GROUPED SPLIT RESULTS")
    print("----------------------------")
    print("Group column:", group_col)
    print("Features:", features)
    print("Training rows:", len(X_train))
    print("Test rows:", len(X_test))
    print(
        "Training groups:",
        groups.iloc[train_idx].nunique()
    )
    print(
        "Test groups:",
        groups.iloc[test_idx].nunique()
    )

    print("\nMeasured results:")
    print(
        "Accuracy :",
        round(
            accuracy_score(y_test, predictions),
            4
        )
    )

    print(
        "Precision:",
        round(
            precision_score(
                y_test,
                predictions,
                zero_division=0
            ),
            4
        )
    )

    print(
        "Recall   :",
        round(
            recall_score(
                y_test,
                predictions,
                zero_division=0
            ),
            4
        )
    )

    print(
        "F1       :",
        round(
            f1_score(
                y_test,
                predictions,
                zero_division=0
            ),
            4
        )
    )

Dataset loaded successfully
Rows: 30000
Columns: 44

HONEST GROUPED SPLIT RESULTS
----------------------------
Group column: client_id
Features: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'search_volume', 'word_count', 'position_tier']
Training rows: 23837
Test rows: 6163
Training groups: 25
Test groups: 7

Measured results:
Accuracy : 0.5682
Precision: 0.5616
Recall   : 0.7066
F1       : 0.6258



## 3. Leakage audit

I repeat the Week-3 leakage hunt on the final feature set. I check whether any feature directly contains the target, a future outcome, or information that would only be available after the prediction point. Any suspicious feature is treated cautiously. The goal is to keep the final model useful for decision-support rather than accidentally using future information.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 3: basic leakage audit

print("Final feature set leakage audit")

feature_names = list(X.columns) if hasattr(X, "columns") else []

target_name = getattr(y, "name", None)

print("Number of features:", len(feature_names))
print("Target:", target_name)

# Check feature names for obvious target/leakage words
leakage_words = [
    "target", "label", "outcome", "future",
    "next", "after", "conversion", "result"
]

suspicious = [
    col for col in feature_names
    if any(word in str(col).lower() for word in leakage_words)
]

print("\nPotentially suspicious features:")
print(suspicious if suspicious else "None found from feature-name check.")

# Exact target-name check
if target_name is not None and target_name in feature_names:
    print("\nWARNING: target column is present in features.")
else:
    print("\nTarget-column check: PASS")

Final feature set leakage audit
Number of features: 6
Target: is_declining_label

Potentially suspicious features:
None found from feature-name check.

Target-column check: PASS



## 4. Claim rewrite

### Original bold claim
The model can reliably predict content performance and identify which content should be refreshed.

### Safer claim
The measured results provide directional evidence about patterns associated with content outcomes. Under the tested validation design, the model can be used as decision-support for prioritizing content for further review, but the results should not be interpreted as proof of causation or guaranteed future performance.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: claim audit

original_claim = (
    "The model can reliably predict content performance and identify "
    "which content should be refreshed."
)

safe_claim = (
    "The measured results provide directional evidence about patterns "
    "associated with content outcomes. Under the tested validation design, "
    "the model can be used as decision-support for prioritizing content "
    "for further review, but it does not prove causation or guarantee "
    "future performance."
)

print("Original claim:")
print(original_claim)

print("\nSafer claim:")
print(safe_claim)

print("\nClaim language check:")
required_words = ["measured", "directional", "decision-support"]

for word in required_words:
    print(f"{word}: {word in safe_claim}")

Original claim:
The model can reliably predict content performance and identify which content should be refreshed.

Safer claim:
The measured results provide directional evidence about patterns associated with content outcomes. Under the tested validation design, the model can be used as decision-support for prioritizing content for further review, but it does not prove causation or guarantee future performance.

Claim language check:
measured: True
directional: True
decision-support: True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.